## Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import glob
import re
import os

## Load the CSV Data

In [2]:
all_csv_files = glob.glob("../data_in_buckets/*.csv")

dfs = []
for f in all_csv_files:
    df_temp = pd.read_csv(f)
    df_temp["game_id"] = os.path.splitext(os.path.basename(f))[0]
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)

Let's just view the head of the dataframe to make sure the data was read correctly.

In [3]:
df.head()

,timestamp,comment_count,avg_score,all_text,offset_from_kickoff_min,timestamp_et,week,game,final_score,winner,thread_url,game_id
0,2025-12-07 16:54:00+00:00,1,2.0,Jayden rubber bones back in action,-36.0,2025-12-07 11:54:00,14,Commanders (WAS) at Vikings (MIN),"=IFERROR(__xludf.DUMMYFUNCTION(""AI(""""what was ...","=IFERROR(__xludf.DUMMYFUNCTION(""AI(""""Fill an a...",https://www.reddit.com/r/nfl/comments/1pgn4c4/...,week14_Commanders__WAS__at_Vikings__MIN_buckets
1,2025-12-07 16:54:30+00:00,1,4.0,Jalyn Holmes is still in the league? Good for ...,-35.5,2025-12-07 11:54:30,14,Commanders (WAS) at Vikings (MIN),"=IFERROR(__xludf.DUMMYFUNCTION(""AI(""""what was ...","=IFERROR(__xludf.DUMMYFUNCTION(""AI(""""Fill an a...",https://www.reddit.com/r/nfl/comments/1pgn4c4/...,week14_Commanders__WAS__at_Vikings__MIN_buckets
2,2025-12-07 16:57:00+00:00,2,4.0,Thought he was a rookie for us last year 😭 had...,-33.0,2025-12-07 11:57:00,14,Commanders (WAS) at Vikings (MIN),"=IFERROR(__xludf.DUMMYFUNCTION(""AI(""""what was ...","=IFERROR(__xludf.DUMMYFUNCTION(""AI(""""Fill an a...",https://www.reddit.com/r/nfl/comments/1pgn4c4/...,week14_Commanders__WAS__at_Vikings__MIN_buckets
3,2025-12-07 16:58:30+00:00,2,4.0,Nine ve JD5 🍿🍿🍿🍿🍿\n\nQB1 matchup from the Clas...,-31.5,2025-12-07 11:58:30,14,Commanders (WAS) at Vikings (MIN),"=IFERROR(__xludf.DUMMYFUNCTION(""AI(""""what was ...","=IFERROR(__xludf.DUMMYFUNCTION(""AI(""""Fill an a...",https://www.reddit.com/r/nfl/comments/1pgn4c4/...,week14_Commanders__WAS__at_Vikings__MIN_buckets
4,2025-12-07 16:59:00+00:00,1,5.0,This thread should have started 60 minutes lat...,-31.0,2025-12-07 11:59:00,14,Commanders (WAS) at Vikings (MIN),"=IFERROR(__xludf.DUMMYFUNCTION(""AI(""""what was ...","=IFERROR(__xludf.DUMMYFUNCTION(""AI(""""Fill an a...",https://www.reddit.com/r/nfl/comments/1pgn4c4/...,week14_Commanders__WAS__at_Vikings__MIN_buckets


## Format Timestamps

In [5]:
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df["timestamp_et"] = pd.to_datetime(df["timestamp_et"])

Once again, let's just view the head of the dataframe to make sure the timestamps look correct.

In [6]:
df.head()

,timestamp,comment_count,avg_score,all_text,offset_from_kickoff_min,timestamp_et,week,game,thread_url,game_id
0,2025-12-07 16:54:00+00:00,1,2.0,Jayden rubber bones back in action,-36.0,2025-12-07 11:54:00,14,Commanders (WAS) at Vikings (MIN),https://www.reddit.com/r/nfl/comments/1pgn4c4/...,week14_Commanders__WAS__at_Vikings__MIN_buckets
1,2025-12-07 16:54:30+00:00,1,4.0,Jalyn Holmes is still in the league? Good for ...,-35.5,2025-12-07 11:54:30,14,Commanders (WAS) at Vikings (MIN),https://www.reddit.com/r/nfl/comments/1pgn4c4/...,week14_Commanders__WAS__at_Vikings__MIN_buckets
2,2025-12-07 16:57:00+00:00,2,4.0,Thought he was a rookie for us last year 😭 had...,-33.0,2025-12-07 11:57:00,14,Commanders (WAS) at Vikings (MIN),https://www.reddit.com/r/nfl/comments/1pgn4c4/...,week14_Commanders__WAS__at_Vikings__MIN_buckets
3,2025-12-07 16:58:30+00:00,2,4.0,Nine ve JD5 🍿🍿🍿🍿🍿\n\nQB1 matchup from the Clas...,-31.5,2025-12-07 11:58:30,14,Commanders (WAS) at Vikings (MIN),https://www.reddit.com/r/nfl/comments/1pgn4c4/...,week14_Commanders__WAS__at_Vikings__MIN_buckets
4,2025-12-07 16:59:00+00:00,1,5.0,This thread should have started 60 minutes lat...,-31.0,2025-12-07 11:59:00,14,Commanders (WAS) at Vikings (MIN),https://www.reddit.com/r/nfl/comments/1pgn4c4/...,week14_Commanders__WAS__at_Vikings__MIN_buckets


## Drop Rows w/No Text

In [7]:
df = df.dropna(subset=["all_text"])
df = df[df["all_text"].str.strip() != ""]

## Remove Any Deleted Content

In [8]:
df = df[~df["all_text"].str.fullmatch(r'[\s\|\[deleted\]\[removed\]]+', na=False)]
df["all_text"] = df["all_text"].str.replace(r'\[deleted\]|\[removed\]', '', regex=True).str.strip()

print(df.head())

                  timestamp  comment_count  avg_score  \
0 2025-12-07 16:54:00+00:00              1        2.0   
1 2025-12-07 16:54:30+00:00              1        4.0   
2 2025-12-07 16:57:00+00:00              2        4.0   
3 2025-12-07 16:58:30+00:00              2        4.0   
4 2025-12-07 16:59:00+00:00              1        5.0   

                                            all_text  offset_from_kickoff_min  \
0                 Jayden rubber bones back in action                    -36.0   
1  Jalyn Holmes is still in the league? Good for ...                    -35.5   
2  Thought he was a rookie for us last year 😭 had...                    -33.0   
3  Nine ve JD5 🍿🍿🍿🍿🍿\n\nQB1 matchup from the Clas...                    -31.5   
4  This thread should have started 60 minutes lat...                    -31.0   

         timestamp_et  week                               game  \
0 2025-12-07 11:54:00    14  Commanders (WAS) at Vikings (MIN)   
1 2025-12-07 11:54:30    14  Commander

## Clean the Comments (formatting, weird symbols, etc)

In [9]:
def clean_text(text):
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[*~#]+', '', text)
    text = text.replace(' | ', ' </s> ')  # replace pipe separator with RoBERTa separator token
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["all_text_clean"] = df["all_text"].apply(clean_text)

Quick test on cleaned up data:

In [10]:
print("BEFORE:", df["all_text"].iloc[2])
print("AFTER:", df["all_text_clean"].iloc[2])

BEFORE: Thought he was a rookie for us last year 😭 had no idea he been in the league that long | If JJM can just look mid I will be elated
AFTER: Thought he was a rookie for us last year 😭 had no idea he been in the league that long </s> If JJM can just look mid I will be elated


## Create a List of Comments

Split each bucket's concatenated text into a list of individual comments using the `</s>` separator token.
This preserves the comment boundaries for use with hierarchical attention if needed.

(Yes, admittedly, the </s> token was probably not needed in thiscase)

In [ ]:
df["comments_list"] = df["all_text_clean"].str.split(r'\s*</s>\s*')
print(df["comments_list"].iloc[2]) # just a random entry in the dataset that I found with 2 comments

['Thought he was a rookie for us last year 😭 had no idea he been in the league that long', 'If JJM can just look mid I will be elated']


## Deduplicate and Sort Chronologically

Remove duplicate timestamps within the same game before gap-filling.
Keep the row with the highest comment count when duplicates exist.
Then sort so every game is in strict chronological order.

In [12]:
df = df.sort_values(["game_id", "timestamp", "comment_count"], ascending=[True, True, False])
df = df.drop_duplicates(subset=["game_id", "timestamp"], keep="first")
df = df.reset_index(drop=True)

## Handle Missing Buckets/Gaps in Time

Create a complete 30-second grid for each game from first to last timestamp.
Fill empty buckets with empty string for text and 0 for numeric columns.
This ensures the model sees a regular sequence with gaps represented as silence.

In [13]:
def fill_missing_buckets(game_df):
    full_range = pd.date_range(
        start=game_df["timestamp"].min(),
        end=game_df["timestamp"].max(),
        freq="30s",
        tz="UTC"
    )
    game_df = game_df.set_index("timestamp").reindex(full_range)
    game_df.index.name = "timestamp"

    game_df[["game_id", "week", "game", "offset_from_kickoff_min"]] = \
        game_df[["game_id", "week", "game", "offset_from_kickoff_min"]].ffill()

    game_df["all_text_clean"] = game_df["all_text_clean"].fillna("")
    game_df["comment_count"] = game_df["comment_count"].fillna(0)
    game_df["avg_score"] = game_df["avg_score"].fillna(0)

    return game_df.reset_index()

df = df.groupby("game_id", group_keys=False).apply(fill_missing_buckets)

## Recompute Offsets from Kickoff

The original `offset_from_kickoff_min` values are corrupted in several CSVs. I'm just recomputing them from scratch here so we don't have NaNs

In [14]:
kickoff_times = (df.groupby("game_id")
                 .apply(lambda g: g.loc[(g["offset_from_kickoff_min"] - 0).abs().idxmin(), "timestamp"])
                 .rename("kickoff_time"))

df = df.join(kickoff_times, on="game_id")
df["offset_from_kickoff_min"] = (
    (df["timestamp"] - df["kickoff_time"])
    .dt.total_seconds() / 60
)
df = df.drop(columns=["kickoff_time"])

## Clip to a Reasonable Time Range

Keep only rows within 60 minutes before kickoff and 360 minutes after.
This removes corrupted timestamps from bad CSVs and postgame noise.

IMPORTANT: this is likely something that needs to be tuned once all the Kalshi data is obtained!

In [15]:
df = df[
    (df["offset_from_kickoff_min"] >= -60) &
    (df["offset_from_kickoff_min"] <= 360)
]

## Fix Null Entries and Recompute Some Columns

After clipping from earlier, we can recompute `is_pregame`. Also forward-filling the `thread_url` within each game for gap-filled rows and recomputing `timestamp_et` from the UTC timestamp directly.

In [ ]:
df["is_pregame"] = df["offset_from_kickoff_min"] < 0
df["timestamp_et"] = pd.to_datetime(df["timestamp"]).dt.tz_convert("US/Eastern")
df["thread_url"] = df.groupby("game_id")["thread_url"].ffill()

## Sanity Check

In [ ]:
print(f"Total rows: {len(df)}")
print(f"Total games: {df['game_id'].nunique()}")
print(f"Rows with empty text: {(df['all_text_clean'] == '').sum()}")
print(f"Offset range: {df['offset_from_kickoff_min'].min():.1f} to {df['offset_from_kickoff_min'].max():.1f} min")
print(f"\nSample cleaned text:")
print(df[df['all_text_clean'] != '']['all_text_clean'].iloc[0])
print(f"Pregame rows: {df['is_pregame'].sum()}")
print(f"Avg pregame buckets per game: {df[df['is_pregame']]['game_id'].value_counts().mean():.1f}")
print(f"Columns: {df.columns.tolist()}")
print(f"Any nulls? \n{df.isnull().sum()}")

Total rows: 118877
Total games: 215
Rows with empty text: 42773
Offset range: -51.5 to 360.0 min

Sample cleaned text:
I’m gonna be clenched every time CMC touches the ball.
Pregame rows: 12419
Avg pregame buckets per game: 66.1
Columns: ['timestamp', 'comment_count', 'avg_score', 'all_text', 'offset_from_kickoff_min', 'timestamp_et', 'week', 'game', 'thread_url', 'game_id', 'all_text_clean', 'comments_list', 'is_pregame']
Any nulls:
timestamp                      0
comment_count                  0
avg_score                      0
all_text                   42767
offset_from_kickoff_min        0
timestamp_et                   0
week                           0
game                           0
thread_url                     0
game_id                        0
all_text_clean                 0
comments_list              42767
is_pregame                     0
dtype: int64


## Save Checkpoint for later preprocessing work

In [18]:
df.to_parquet("reddit_cleaned.parquet", index=False)
print("Saved successfully.")

Saved successfully.


## Another sanity check, just obtaining the list of games

In [19]:
df = pd.read_parquet("reddit_cleaned.parquet")
game_list = df[["week", "game", "game_id"]].drop_duplicates("game_id").sort_values(["week", "game"])
print(game_list.to_string())
print(len(game_list))

        week                                         game                                                    game_id
0        1.0                 49ers (SF) at Seahawks (SEA)                 week01_49ers__SF__at_Seahawks__SEA_buckets
799      1.0                Bengals (CIN) at Browns (CLE)                week01_Bengals__CIN__at_Browns__CLE_buckets
1580     1.0             Buccaneers (TB) at Falcons (ATL)             week01_Buccaneers__TB__at_Falcons__ATL_buckets
2353     1.0               Cardinals (ARI) at Saints (NO)               week01_Cardinals__ARI__at_Saints__NO_buckets
2855     1.0                Cowboys (DAL) at Eagles (PHI)                week01_Cowboys__DAL__at_Eagles__PHI_buckets
3098     1.0                Dolphins (MIA) at Colts (IND)                week01_Dolphins__MIA__at_Colts__IND_buckets
3602     1.0             Giants (NYG) at Commanders (WAS)             week01_Giants__NYG__at_Commanders__WAS_buckets
4401     1.0                  Lions (DET) at Packers (GB)       

## Wrong Quantization
We wrongly assumed Kalshi provides data in 30-second intervals. It's actually 1 minute. So, we need to adjust our buckets accordingly.

In [ ]:
df = pd.read_parquet("reddit_cleaned.parquet")

def resample_to_1min(game_df):
    game_df = game_df.set_index("timestamp").sort_index()
    resampled = pd.DataFrame()
    resampled["comment_count"] = game_df["comment_count"].resample("1min").sum()

    def weighted_avg(g):
        total = g["comment_count"].sum()
        return (g["comment_count"] * g["avg_score"]).sum() / total if total > 0 else 0.0
    resampled["avg_score"] = game_df[["comment_count", "avg_score"]].resample("1min").apply(weighted_avg)

    resampled["all_text_clean"] = game_df["all_text_clean"].resample("1min").apply(
        lambda texts: " </s> ".join(t for t in texts if t != "")
    )
    resampled["comments_list"] = resampled["all_text_clean"].apply(
        lambda t: [c.strip() for c in t.split(" </s> ") if c.strip()] if t else []
    )

    for col in ["game_id", "week", "game", "thread_url",
                "offset_from_kickoff_min", "is_pregame"]:
        resampled[col] = game_df[col].resample("1min").first()

    return resampled.reset_index()

df_1min = df.groupby("game_id", group_keys=False).apply(resample_to_1min)
df_1min.to_parquet("reddit_cleaned_1min.parquet", index=False)

In [24]:
df_1min.to_csv("reddit_cleaned_1min.csv", index=False) # converting to CSV to visually inspect